# One GRPO Step on One SWE-bench Task

**Goal: conceptual clarity, not training.** Nothing here converges. By the end
you will have watched a single gradient update travel the entire pipeline, with
every intermediate value printed.

The five stages, and nothing else:

1. Load **one** instance from SWE-bench Verified
2. A minimal mini-swe-agent-shaped harness produces rollouts
3. Every bash call is **mocked** — no Docker, no repo clone, no execution
4. The final patch gets a reward
5. A group of rewards becomes **one** GRPO step

### The one thing to hold onto

You can mock the filesystem. You can mock `grep`, `cat`, and `pytest`. You
cannot mock the reward — a real reward requires really running the tests in a
real environment. Section 4 is where the pretending stops, and §6 keeps the
ledger honest.

Runs on a T4 in a few minutes. Also runs on CPU if you're patient.

In [1]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" torch

import torch, re, json, difflib, random
import numpy as np
import pandas as pd
from IPython.display import display

random.seed(0); torch.manual_seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


device: cuda


---
# 1. Load one instance

500 human-validated tasks. We take exactly one.

In [2]:
from datasets import load_dataset

ds = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")
print(ds)

# A small, single-file task keeps the walkthrough readable.
cands = [i for i, r in enumerate(ds)
         if r["patch"].count("diff --git") == 1 and len(r["patch"]) < 1800]
print(f"\n{len(cands)} single-file instances with a short gold patch")

inst = ds[cands[0]]
for k in ["instance_id", "repo", "base_commit", "version", "difficulty"]:
    print(f"{k:20} {inst.get(k)}")

/home/agoswami/miniconda3/envs/swe_grpo_vizuara_01/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'difficulty'],
    num_rows: 500
})

380 single-file instances with a short gold patch
instance_id          astropy__astropy-12907
repo                 astropy/astropy
base_commit          d16bfe05a744909de4b27f5875fe0d4ed41ce607
version              4.3
difficulty           15 min - 1 hour


In [3]:
print("========== problem_statement (the issue text the agent sees) ==========")
print(inst["problem_statement"][:1200])
print("\n... [truncated]")

========== problem_statement (the issue text the agent sees) ==========
Modeling's `separability_matrix` does not compute separability correctly for nested CompoundModels
Consider the following model:

```python
from astropy.modeling import models as m
from astropy.modeling.separable import separability_matrix

cm = m.Linear1D(10) & m.Linear1D(5)
```

It's separability matrix as you might expect is a diagonal:

```python
>>> separability_matrix(cm)
array([[ True, False],
       [False,  True]])
```

If I make the model more complex:
```python
>>> separability_matrix(m.Pix2Sky_TAN() & m.Linear1D(10) & m.Linear1D(5))
array([[ True,  True, False, False],
       [ True,  True, False, False],
       [False, False,  True, False],
       [False, False, False,  True]])
```

The output matrix is again, as expected, the outputs and inputs to the linear models are separable and independent of each other.

If however, I nest these compound models:
```python
>>> separability_matrix(m.Pix2Sky_TAN() 

In [4]:
print("========== patch (GOLD — the agent never sees this) ==========")
print(inst["patch"])

print("\n========== FAIL_TO_PASS (tests that must start passing) ==========")
print(json.dumps(json.loads(inst["FAIL_TO_PASS"])[:5], indent=1))
p2p = json.loads(inst["PASS_TO_PASS"])
print(f"\nPASS_TO_PASS: {len(p2p)} tests that must KEEP passing")
print(json.dumps(p2p[:3], indent=1))

========== patch (GOLD — the agent never sees this) ==========
diff --git a/astropy/modeling/separable.py b/astropy/modeling/separable.py
--- a/astropy/modeling/separable.py
+++ b/astropy/modeling/separable.py
@@ -242,7 +242,7 @@ def _cstack(left, right):
         cright = _coord_matrix(right, 'right', noutp)
     else:
         cright = np.zeros((noutp, right.shape[1]))
-        cright[-right.shape[0]:, -right.shape[1]:] = 1
+        cright[-right.shape[0]:, -right.shape[1]:] = right
 
     return np.hstack([cleft, cright])
 


========== FAIL_TO_PASS (tests that must start passing) ==========
[
 "astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]",
 "astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]"
]

PASS_TO_PASS: 13 tests that must KEEP passing
[
 "astropy/modeling/tests/test_separable.py::test_coord_matrix",
 "astropy/modeling/tests/test_separable.py::test_cdot",
 "astropy/modeling/tests/test_separable.py::test_cstac

**This is the whole task definition.** An issue, a commit, a gold patch, and two
lists of test names. Notice what is *not* here: the repository. Real evaluation
clones it at `base_commit`, applies `test_patch`, applies the candidate patch,
and runs both test lists. That's the part needing Docker, and the part we mock.

`PASS_TO_PASS` is the anti-reward-hacking mechanism. Without it, deleting the
failing test would score a win.

---
# 2. Reconstruct a mock repo from the gold patch

The trick that makes this notebook possible offline.

A unified diff contains three kinds of lines: `-` (removed), `+` (added), and
` ` (context). Take the `-` and ` ` lines and you have **the real pre-fix source
of that region.** Not invented — genuinely from the repo, just partial.

So the gold patch gives us a small but authentic mock filesystem for free.

In [5]:
def parse_patch(patch):
    '''-> {path: {"pre": [lines], "post": [lines], "added": [str], "removed": [str]}}'''
    files, cur = {}, None
    for line in patch.split("\n"):
        if line.startswith("diff --git"):
            # NB: don't regex for "b/" - it also matches inside paths like
            # django/db/... . Take the last whitespace-separated token.
            last = line.split()[-1]
            cur = last[2:] if last.startswith("b/") else None
            if cur:
                files[cur] = dict(pre=[], post=[], added=[], removed=[])
        elif cur is None or line.startswith(("index ", "--- ", "+++ ")):
            continue
        elif line.startswith("@@"):
            continue                            # drop hunk headers: they would
                                                # pollute the candidate diff
        elif line.startswith("+"):
            files[cur]["post"].append(line[1:])
            files[cur]["added"].append(line[1:].strip())
        elif line.startswith("-"):
            files[cur]["pre"].append(line[1:])
            files[cur]["removed"].append(line[1:].strip())
        elif line.startswith(" "):
            files[cur]["pre"].append(line[1:])
            files[cur]["post"].append(line[1:])
    return files

GOLD = parse_patch(inst["patch"])
TARGET = list(GOLD)[0]

print("files touched by gold patch:", list(GOLD))
print(f"\nreconstructed pre-fix view of {TARGET}:\n")
print("\n".join(GOLD[TARGET]["pre"]))
print("\n--- gold ADDS these lines ---")
for a in GOLD[TARGET]["added"]:
    print("  +", a)

files touched by gold patch: ['astropy/modeling/separable.py']

reconstructed pre-fix view of astropy/modeling/separable.py:

        cright = _coord_matrix(right, 'right', noutp)
    else:
        cright = np.zeros((noutp, right.shape[1]))
        cright[-right.shape[0]:, -right.shape[1]:] = 1

    return np.hstack([cleft, cright])


--- gold ADDS these lines ---
  + cright[-right.shape[0]:, -right.shape[1]:] = right


---
# 3. A minimal harness with mocked tools

Shaped after mini-swe-agent: **bash only**, no tool-calling API, a strictly
linear message history, one command per turn. The difference is that
`subprocess.run` is replaced by a lookup table.

The mock handles `ls`, `cat`, `grep`, `pytest`, and a heredoc write. Anything
else returns a stub. That's the point — you can see exactly how thin the
environment is.

In [6]:
SYSTEM = f'''You are a software engineering agent. You are in a Python repository.
Fix the bug described in the issue.

Respond with exactly ONE bash command per message, in a fenced block:

```bash
your command here
```

Useful commands:
  ls
  cat {TARGET}
  grep -n "pattern" {TARGET}
  python -m pytest

To rewrite a file:
```bash
cat > {TARGET} <<'EOF'
...full new contents...
EOF
```

One short sentence of reasoning, then exactly one command block.'''

RE_BASH = re.compile(r"```(?:bash|sh)?\s*\n(.*?)```", re.S)
RE_HEREDOC = re.compile(r"cat\s*>\s*(\S+)\s*<<\s*'?EOF'?\n(.*?)\nEOF", re.S)


class MockEnv:
    '''Every method here is a lie. The docstrings say which kind.'''

    def __init__(self, gold, target):
        # Virtual FS seeded with the REAL pre-fix source recovered from the patch.
        self.fs = {p: "\n".join(v["pre"]) for p, v in gold.items()}
        self.orig = dict(self.fs)
        self.target = target
        self.calls = []

    def run(self, cmd):
        self.calls.append(cmd)

        m = RE_HEREDOC.search(cmd)
        if m:                                   # real effect on the virtual FS
            path, body = m.group(1), m.group(2)
            self.fs[path] = body
            return f"[mock] wrote {len(body.splitlines())} lines to {path}"

        if cmd.strip().startswith("ls"):        # canned
            return "\n".join(sorted(self.fs)) + "\nsetup.py\nREADME.rst\ntests/"

        if cmd.strip().startswith("cat "):      # real data, partial view
            path = cmd.split()[-1]
            if path in self.fs:
                return (f"[mock: only the region near the bug is available]\n"
                        + self.fs[path])
            return f"cat: {path}: No such file or directory"

        if cmd.strip().startswith("grep"):      # real search over the fragment
            pat = re.findall(r'"([^"]*)"|\'([^\']*)\'', cmd)
            pat = next((a or b for a, b in pat), "")
            hits = [f"{i+1}:{l}" for i, l in enumerate(self.fs[self.target].split("\n"))
                    if pat and pat in l]
            return "\n".join(hits) if hits else "(no matches)"

        if "pytest" in cmd:                     # canned failure, real test names
            names = json.loads(inst["FAIL_TO_PASS"])[:2]
            body = "\n".join(f"FAILED {n}" for n in names)
            return f"[mock] {body}\n=== {len(names)} failed ===\n(mock never actually runs anything)"

        return f"[mock] '{cmd.split()[0]}' not implemented in this stub"

    def patch(self):
        '''The candidate patch: a real unified diff of the virtual FS.'''
        out = []
        for p in self.fs:
            if self.fs[p] != self.orig[p]:
                out += list(difflib.unified_diff(
                    self.orig[p].split("\n"), self.fs[p].split("\n"),
                    fromfile=f"a/{p}", tofile=f"b/{p}", lineterm=""))
        return "\n".join(out)


env = MockEnv(GOLD, TARGET)
print(env.run("ls"))
print()
print(env.run("python -m pytest")[:300])

astropy/modeling/separable.py
setup.py
README.rst
tests/

[mock] FAILED astropy/modeling/tests/test_separable.py::test_separable[compound_model6-result6]
FAILED astropy/modeling/tests/test_separable.py::test_separable[compound_model9-result9]
=== 2 failed ===
(mock never actually runs anything)


In [7]:
def rollout(model, tok, max_turns=4, max_new=200, temperature=1.0):
    '''mini-swe-agent shape: linear history, one bash command per turn.'''
    env = MockEnv(GOLD, TARGET)
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"ISSUE:\n{inst['problem_statement'][:1500]}"}]

    for _ in range(max_turns):
        prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ip = tok(prompt, return_tensors="pt", truncation=True, max_length=3072).to(DEV)
        with torch.no_grad():
            out = model.generate(**ip, max_new_tokens=max_new,
                                 do_sample=temperature > 0,
                                 temperature=max(temperature, 1e-5), top_p=0.95,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        reply = tok.decode(out[0][ip["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        msgs.append({"role": "assistant", "content": reply})

        m = RE_BASH.search(reply)
        obs = env.run(m.group(1).strip()) if m else \
              "No command found. Reply with exactly one ```bash block."
        msgs.append({"role": "user", "content": obs[:800]})

    return dict(messages=msgs, patch=env.patch(), final=dict(env.fs), calls=env.calls)

---
# 4. Reward

Here is where mocking runs out.

**The real reward** clones the repo at `base_commit`, applies `test_patch`, applies
the candidate patch, runs `FAIL_TO_PASS` and `PASS_TO_PASS`, and returns
`1.0` only if every test in both lists passes. Binary. No partial credit.

**Our mock reward** compares the candidate patch's added lines to the gold
patch's added lines. That is a *proxy*, and a leaky one: it rewards reproducing
the maintainer's exact fix rather than any fix that works. Real SWE-bench
deliberately does not do this, because there are many correct patches.

We compute both, and the difference between them is the lesson.

In [8]:
GOLD_POST  = "\n".join(GOLD[TARGET]["post"])

def norm_lines(text):
    return {re.sub(r"\s+", " ", l).strip() for l in text.split("\n") if l.strip()}

GOLD_POST_LINES = norm_lines(GOLD_POST)

def reward_binary(final):
    '''Mock of the real reward. Did the file end up as the maintainer left it?'''
    return float(norm_lines(final.get(TARGET, "")) == GOLD_POST_LINES)

def reward_random(n, seed=0):
    '''MOCK reward: a random stand-in for a real verifier (unit tests / reward model).
    One value per rollout, seeded for reproducibility. NOT a real signal -- it only
    lets us show how the update consumes a reward.'''
    return np.random.default_rng(seed).random(n).round(3)

# sanity check: the real-ish binary reward is 1.0 only for the exact gold file state
pre_state  = {TARGET: "\n".join(GOLD[TARGET]["pre"])}
post_state = {TARGET: GOLD_POST}
print("gold post-state binary:", reward_binary(post_state))
print("untouched file  binary:", reward_binary(pre_state))
print("empty           binary:", reward_binary({}))

gold post-state binary: 1.0
untouched file  binary: 0.0
empty           binary: 0.0


Scoring the **resulting file content**, not the diff text, is deliberate. The
same edit can be written as many different valid diffs, so comparing diff lines
is fragile. Real SWE-bench sidesteps this entirely by applying the patch and
running tests — it never inspects the diff at all.

---
# 5. Sample a group

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16 if DEV == "cuda" else torch.float32,
    attn_implementation="sdpa").to(DEV)

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]))
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3320.88it/s]

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [10]:
G = 6
group = [rollout(model, tok) for _ in range(G)]
mock_reward = reward_random(len(group))   # MOCK verifier: one random reward per rollout

rollouts = pd.DataFrame([
    {
        "rollout":       i,
        "binary_reward": reward_binary(r["final"]),
        "mock_reward":   mock_reward[i],
        "patch_lines":   len(r["patch"].split("\n")) if r["patch"] else 0,
        "n_commands":    len(r["calls"]),
    }
    for i, r in enumerate(group)
])
display(rollouts)

# what one rollout actually did (the mini-swe-agent loop)
print("\nexample -- rollout 0 command trace:")
for cmd in group[0]["calls"]:
    print("  $", cmd.split("\n")[0][:60])

,rollout,binary_reward,mock_reward,patch_lines,n_commands
0,0,0.0,0.637,0,4
1,1,0.0,0.270,0,1
2,2,0.0,0.041,0,3
3,3,0.0,0.017,0,4
4,4,0.0,0.813,0,3
5,5,0.0,0.913,27,3



example -- rollout 0 command trace:
  $ ls | grep -n 'astropy.modeling.separable.py' | awk '{print $
  $ python setup.py install
  $ mock -m unittest.mock -v 3
  $ mock -m unittest.mock -v 3


In [11]:
binary_reward = np.array([reward_binary(r["final"]) for r in group])
mock_reward   = reward_random(len(group))     # same seeded random rewards as above

print("binary_reward:", binary_reward, " std =", binary_reward.std().round(4))
print("mock_reward:  ", mock_reward,   " std =", mock_reward.std().round(4))
print()
if binary_reward.std() < 1e-6:
    print("The binary_reward group is DEGENERATE -- advantage = 0 for every rollout,")
    print("so the gradient would be exactly zero. This is the normal case: a 0.5B")
    print("model on a real SWE-bench task essentially never produces the gold patch.")
    print()
    print("Switching to the mock_reward (random) so there is a signal to show.")
    print("A real setup scores rollouts with unit tests or a reward model;")
    print("random here only demonstrates how the update consumes the reward.")

binary_reward: [0. 0. 0. 0. 0. 0.]  std = 0.0
mock_reward:   [0.637 0.27  0.041 0.017 0.813 0.913]  std = 0.3578

The binary_reward group is DEGENERATE -- advantage = 0 for every rollout,
so the gradient would be exactly zero. This is the normal case: a 0.5B
model on a real SWE-bench task essentially never produces the gold patch.

Switching to the mock_reward (random) so there is a signal to show.
A real setup scores rollouts with unit tests or a reward model;
random here only demonstrates how the update consumes the reward.


---
# 6. One GRPO step

$$A_i = \frac{r_i - \mathrm{mean}(r)}{\mathrm{std}(r) + \varepsilon}$$

No critic. **The other rollouts are the baseline** — that is the entire idea.
Loss is a policy gradient over assistant tokens only; observation tokens came
from the environment, so crediting them is meaningless.

In [12]:
def build_masked(messages, tokenizer, max_len=3072):
    ids, labels, prev = [], [], ""
    for i, m in enumerate(messages):
        cur = tokenizer.apply_chat_template(messages[:i+1], tokenize=False)
        assert cur.startswith(prev), "chat template is not append-only"
        seg = tokenizer(cur[len(prev):], add_special_tokens=False)["input_ids"]
        ids += seg
        labels += seg if m["role"] == "assistant" else [-100]*len(seg)
        prev = cur
    return ids[:max_len], labels[:max_len]


def seq_logprob(messages):
    ids, labs = build_masked(messages, tok)
    t = torch.tensor([ids], device=DEV)
    msk = torch.tensor([[0. if l == -100 else 1. for l in labs]], device=DEV)[:, 1:]
    logits = model(t).logits[:, :-1]
    lp = torch.log_softmax(logits.float(), -1).gather(-1, t[:, 1:].unsqueeze(-1)).squeeze(-1)
    return (lp * msk).sum() / msk.sum().clamp(min=1), msk.sum().item()

In [13]:
rewards = torch.tensor(mock_reward, dtype=torch.float)   # mock random reward, per the note above
adv = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-4)

print(f"mean mock_reward {rewards.mean():.3f}   std {rewards.std(unbiased=False):.3f}")
pd.DataFrame({
    "rollout":     list(range(len(group))),
    "mock_reward": rewards.numpy().round(3),
    "advantage":   adv.numpy().round(3),
    "sup_tokens":  [int(seq_logprob(g["messages"])[1]) for g in group],
})

mean mock_reward 0.448   std 0.358


,rollout,mock_reward,advantage,sup_tokens
0,0,0.637,0.527,82
1,1,0.270,-0.499,426
2,2,0.041,-1.138,253
3,3,0.017,-1.206,135
4,4,0.813,1.018,385
5,5,0.913,1.298,560


In [14]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5)

logp_before = [seq_logprob(g["messages"])[0].item() for g in group]

model.train(); opt.zero_grad(set_to_none=True)
loss_total = 0.0
for g, a in zip(group, adv):
    lp, _ = seq_logprob(g["messages"])
    loss = -(a.to(DEV) * lp) / len(group)      # REINFORCE with a group baseline
    loss.backward()
    loss_total += loss.item()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in model.parameters() if p.requires_grad], 1.0)
opt.step()

logp_after = [seq_logprob(g["messages"])[0].item() for g in group]

print(f"loss {loss_total:+.5f}   grad_norm {grad_norm:.4f}")
pd.DataFrame({
    "rollout":     list(range(len(group))),
    "advantage":   [round(a.item(), 3) for a in adv],
    "logp_before": [round(b, 4) for b in logp_before],
    "logp_after":  [round(c, 4) for c in logp_after],
    "delta":       [round(c - b, 5) for b, c in zip(logp_before, logp_after)],
})

loss -0.07287   grad_norm 0.5629


,rollout,advantage,logp_before,logp_after,delta
0,0,0.527,-2.3340,-2.3365,-0.00252
1,1,-0.499,-1.1735,-1.1739,-0.00036
2,2,-1.138,-1.1900,-1.1910,-0.00107
3,3,-1.206,-1.7507,-1.7552,-0.00449
4,4,1.018,-1.4771,-1.4773,-0.00024
5,5,1.298,-0.6781,-0.6784,-0.00033


**Read the last column.** Rollouts with positive advantage should have gained
log-probability; negative-advantage rollouts should have lost it. That is the
entire mechanism — no value network, no reward model, just *this trajectory
scored better than its siblings, so make it more likely.*

One step. Effect size is tiny, as it should be at lr=1e-5.

---
# 7. The honest ledger

What we faked, and what each shortcut cost:

| Faked | Real version | What the fake hides |
|---|---|---|
| Repo → patch fragment | full clone at `base_commit` | the agent can't explore, so localization — most of the real difficulty — vanishes |
| `cat`/`grep` → dict lookup | shell in a container | no build, no imports, no cross-file reasoning |
| `pytest` → canned string | real suite at real commit | **everything**; see below |
| binary reward → gold-diff overlap | `FAIL_TO_PASS ∧ PASS_TO_PASS` | rewards imitating the maintainer, not fixing the bug |
| 1 task | 500 (Verified) / 50k (SWE-smith) | no generalization claim is possible |
| 1 step | thousands | no learning |

**The reward is the irreducible part.** Mocking the filesystem costs you
fidelity. Mocking the verifier costs you the method itself — a proxy reward is
something the policy will learn to exploit rather than satisfy. This is why
production SWE RL spends its budget on container fleets, and why the verifier,
not the GPU, is usually the bottleneck.

Good closing question for the room: *we mocked everything except the reward and
still saw a gradient — so why can't you train this way?*

---
## Where to go next

- **Lab 1** — trajectory SFT, loss masking, real SWE-smith data
- **Lab 2** — a full GRPO loop with real execution rewards, on a shrunken gym
- Real frameworks: `NovaSky-AI/SkyRL`, `PrimeIntellect-ai/prime-rl`
- The harness this imitates: `SWE-agent/mini-swe-agent` (pin `<2`)